<a href="https://colab.research.google.com/github/Nithya-Alfonisa/AI-Mini-Chatbot/blob/main/PEC%20Con%20Paper%20presentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Lakshigashree/Crime-Hotspot-Prediction.git

In [ ]:
%cd Crime-Hotspot-Prediction/

In [ ]:
!pip install shap lime xgboost
!pip install shap lime xgboost folium scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import shap
import lime
import lime.lime_tabular
import folium
from folium.plugins import HeatMap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# Uploading the file:
from google.colab import files
print("Please upload 'Crime Prediction in Chicago_Dataset.csv' now.")
uploaded = files.upload()

# Load and basic clean
data = pd.read_csv('Crime Prediction in Chicago_Dataset.csv')
print(f"Dataset Loaded: {data.shape[0]} rows")

Please upload 'Crime Prediction in Chicago_Dataset.csv' now.


In [ ]:
# 1. Temporal Processing
# Converting date strings to datetime objects
data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
data.dropna(subset=['Date', 'Latitude', 'Longitude'], inplace=True)

# Cyclical Encoding: Hour 23 and Hour 0 are now adjacent in the model's eyes
data['hour'] = data['Date'].dt.hour
data['hour_sin'] = np.sin(2 * np.pi * data['hour']/24.0)
data['hour_cos'] = np.cos(2 * np.pi * data['hour']/24.0)

# Day and Month features
data['weekday'] = data['Date'].dt.weekday
data['month'] = data['Date'].dt.month

# 2. Spatial Hotspot Clustering
# Clustering latitudes and longitudes into 12 distinct 'Crime Zones'
coords = data[['Latitude', 'Longitude']]
kmeans = KMeans(n_clusters=12, random_state=42, n_init=10).fit(coords)
data['zone_id'] = kmeans.labels_

# 3. Target Definition
# Binary classification: 1 for Violent/Theft (High Risk), 0 for others
data['target'] = np.where(data['Primary Type'].isin(['BATTERY', 'THEFT', 'ROBBERY', 'ASSAULT']), 1, 0)

# Final Feature Selection
features = ['Latitude', 'Longitude', 'hour_sin', 'hour_cos', 'weekday', 'month', 'zone_id']
X = data[features]
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Feature Engineering Complete.")



In [ ]:
# Training XGBoost (Research standard for tabular data)
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Metrics
preds = xgb_model.predict(X_test)
print("--- Model Performance ---")
print(classification_report(y_test, preds))

In [ ]:
def generate_crime_map(df):
    # Center map on Chicago averages
    chicago_map = folium.Map(location=[df.Latitude.mean(), df.Longitude.mean()], zoom_start=11)

    # Filter only High Risk crimes for the heatmap
    heat_data = [[row['Latitude'], row['Longitude']] for _, row in df[df['target']==1].iterrows()]

    HeatMap(heat_data, radius=15, blur=10).add_to(chicago_map)
    return chicago_map

# Display map for a sample of the data
m = generate_crime_map(data.sample(1000))
m.save('crime_analysis_map.html')
m

In [ ]:

# SHAP Global Interpretation
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_test, feature_names=features)

In [ ]:
# To do this for real, you'd download 'chicago_weather.csv' from Kaggle.
# Here is how you merge and engineer it:
def add_weather_features(crime_df, weather_df):
    # Ensure both have a date column in same format
    crime_df['date_only'] = crime_df['Date'].dt.date
    weather_df['date_only'] = pd.to_datetime(weather_df['date']).dt.date

    # Merge
    df = crime_df.merge(weather_df[['date_only', 'temp_high', 'precipitation']], on='date_only', how='left')

    # Feature: 'Is_Extreme_Weather'
    df['extreme_temp'] = np.where((df['temp_high'] < 20) | (df['temp_high'] > 90), 1, 0)
    return df

In [ ]:
# Create dummy coordinates for major Chicago transit hubs (CTA stations)
cta_hubs = np.array([[41.8781, -87.6298], [41.8832, -87.6487]]) # Example coords

def dist_to_nearest_hub(lat, lon, hubs):
    # Simple Euclidean distance (for IEEE, use Haversine for extra points)
    distances = np.sqrt(np.sum((hubs - np.array([lat, lon]))**2, axis=1))
    return np.min(distances)

data['dist_to_transit'] = data.apply(lambda row: dist_to_nearest_hub(row['Latitude'], row['Longitude'], cta_hubs), axis=1)

In [ ]:
!pip install catboost lightgbm

In [ ]:
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# 1. Define Base Learners (Diverse Algorithms)
# Diversity in models helps the stacker learn different patterns
base_models = [
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=42)),
    ('lgbm', LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31, random_state=42)),
    ('cat', CatBoostClassifier(n_estimators=100, learning_rate=0.05, depth=6, verbose=0, random_state=42))
]

# 2. Define the Meta-Learner
# Logistic Regression is the IEEE standard for the 'final_estimator'
# as it prevents overfitting the combined results.
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5  # 5-fold cross-validation during training
)

print("Training the Ensemble Stacked Model... (This may take a minute)")
stacking_model.fit(X_train, y_train)

# 3. Comparison Metrics
stack_preds = stacking_model.predict(X_test)
print(f"Ensemble Accuracy: {accuracy_score(y_test, stack_preds):.4f}")
print(f"Ensemble F1-Score: {f1_score(y_test, stack_preds):.4f}")

In [ ]:
# Select a feature to analyze (e.g., 'hour_sin')
# This shows how the model's output changes with time of day
shap.dependence_plot("hour_sin", shap_values, X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Quick training of baselines for comparison
lr = LogisticRegression().fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=50).fit(X_train, y_train)

models = {'Logistic Regression': lr, 'Random Forest': rf, 'Stacked Ensemble': stacking_model}

print(f"{'Model':<25} | {'Accuracy':<10} | {'F1-Score':<10}")
print("-" * 50)

for name, model in models.items():
    p = model.predict(X_test)
    acc = accuracy_score(y_test, p)
    f1 = f1_score(y_test, p)
    print(f"{name:<25} | {acc:.4f}     | {f1:.4f}")

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

display = PrecisionRecallDisplay.from_estimator(stacking_model, X_test, y_test, name="Stacked Ensemble")
_ = display.ax_.set_title("Precision-Recall Curve: Resource Optimization Analysis")

In [ ]:
# Check accuracy per crime zone
data_test = X_test.copy()
data_test['true'] = y_test
data_test['pred'] = stack_preds

zone_accuracy = data_test.groupby('zone_id').apply(lambda x: accuracy_score(x['true'], x['pred']))
print("Accuracy consistency across urban zones:")
print(zone_accuracy)

In [ ]:
# Create a feature that counts crimes in the same zone in the last 24 hours
# Note: This requires your dataframe to be sorted by Date
data = data.sort_values('Date')
data['prev_24h_crime_count'] = data.groupby('zone_id')['target'].rolling(window=24).sum().reset_index(0, drop=True)
data['prev_24h_crime_count'] = data['prev_24h_crime_count'].fillna(0)

# Add this to your feature list
features.append('prev_24h_crime_count')

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

prob_true, prob_pred = calibration_curve(y_test, stacking_model.predict_proba(X_test)[:,1], n_bins=10)

plt.plot(prob_pred, prob_true, marker='o', label='Stacked Ensemble')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated')
plt.title("Reliability Curve: Trustworthiness of Predictions")
plt.legend()
plt.show()

In [ ]:
# Select one specific test case (e.g., index 10)
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[10,:], X_test.iloc[10,:])

In [ ]:
# Sort by date to ensure rolling window doesn't leak future data
data = data.sort_values('Date')

# Calculate 'Near-Repeat' lag: Total crimes in that zone in the previous 24 hours
# We shift(1) to ensure we aren't including the current crime in its own feature
data['crimes_last_24h'] = data.groupby('zone_id')['target'].transform(
    lambda x: x.rolling(window=24, min_periods=1).sum().shift(1)
).fillna(0)

# Update your feature list
features = ['Latitude', 'Longitude', 'hour_sin', 'hour_cos', 'weekday', 'month', 'zone_id', 'crimes_last_24h']
X = data[features]
y = data['target']

In [ ]:
def generate_dispatch_note(sample_index):
    # Get SHAP values for a specific high-risk prediction
    val = shap_values[sample_index]
    feat_names = X_test.columns

    # Find top 3 features that pushed the probability UP
    top_contributors = sorted(zip(feat_names, val), key=lambda x: x[1], reverse=True)[:3]

    print(f"--- PREDICTIVE DISPATCH NOTE (Index: {sample_index}) ---")
    print(f"RISK LEVEL: HIGH ({xgb_model.predict_proba(X_test)[sample_index][1]:.2%})")
    print("KEY RISK FACTORS:")
    for feat, impact in top_contributors:
        print(f" - {feat.replace('_', ' ').title()}")
    print("ACTION: Deploy unit for preventative patrol.")

# Run for a sample case
generate_dispatch_note(10)

In [ ]:
from sklearn.metrics import confusion_matrix

results_df = X_test.copy()
results_df['true'] = y_test
results_df['pred'] = stack_preds

def calculate_fpr(group):
    tn, fp, fn, tp = confusion_matrix(group['true'], group['pred'], labels=[0, 1]).ravel()
    return fp / (fp + tn) if (fp + tn) > 0 else 0

fpr_per_zone = results_df.groupby('zone_id').apply(calculate_fpr)

# Visualization for Slides
plt.figure(figsize=(10,4))
fpr_per_zone.plot(kind='bar', color='teal')
plt.title("Fairness Audit: False Positive Rate per Urban Zone")
plt.ylabel("FPR (Bias Metric)")
plt.xlabel("City Zone ID")
plt.axhline(y=fpr_per_zone.mean(), color='r', linestyle='--', label='Average FPR')
plt.legend()
plt.show()

In [ ]:
def check_model_health(current_f1, threshold=0.55):
    if current_f1 < threshold:
        print("🚨 ALERT: Concept Drift Detected!")
        print("Action: Triggering Automated Retraining on most recent 90-day window.")
        # In a real system, you would call: model.fit(X_recent, y_recent)
    else:
        print("✅ Model Health: Stable. No retraining required.")

# Use the F1 score from your stacking model
current_f1 = f1_score(y_test, stack_preds)
check_model_health(current_f1)